# D3 Medellín — Unión del grafo (D2M) con la pendiente (frente de terreno)

Mismo procedimiento que `D3_union_pendiente.ipynb` de Bogotá (las 11 pruebas del contrato),
con lo que cambia en Medellín:

- **Insumos:** `entregables/D2_medellin/` (grafo D2M) y el CSV del módulo de terreno en `datos_crudos/recibidos/`.
- **Salidas:** `entregables/D3_medellin/rama_<nombre>/`. No toca nada de Bogotá ni de D2M.
- **Rama oficial:** σz = 0,539 m (9 vértices del IGAC) y ancho de correlación 59,2 m
  (medido con el método de Bogotá); σz = 0,1 m de la ficha queda como sensibilidad. Se declara en `constantes_d1.RAMAS_PENDIENTE_MED`.. Se declara en `constantes_d1.RAMAS_PENDIENTE_MED`.
- **MDT en EPSG:6257** (grilla urbana de Medellín); la red está en EPSG:9377.
- **Túneles** (`es_tunel`): pendiente de boca a boca, dos puntos, acumulada = |neta|.
  Dos pruebas nuevas lo verifican (12 y 13).
- **Diagnóstico para revisión** (celda final, no decide nada): extremas sin jerarquía,
  `paso_nivel = 2`, elevados y túneles.

In [ ]:
# =============================================================================
# D3M — CELDA 1: insumos y verificacion
# Lee de disco. Escribe SOLO en entregables/D3_medellin. No modifica D2M.
# =============================================================================
import json, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import geopandas as gpd

import constantes_d1 as K
import d2_medellin as d2m          # misma huella con la que D2M registro el grafo

SELLO = datetime.now(ZoneInfo("America/Bogota")).strftime("%Y%m%d_%H%M%S")

def _a_bool(s):
    """Booleanos leidos de CSV/GPKG ('True'/'False', 1/0, bool) -> bool o NaN."""
    return s.map({True: True, False: False, "True": True, "False": False,
                  "true": True, "false": False, 1: True, 0: False})

def guardar_sin_sobrescribir(contenido: bytes, ruta):
    """Escribe solo si no existe. Si existe igual, no lo toca; si existe
    distinto, se detiene (nada se reescribe en silencio)."""
    nuevo = hashlib.sha256(contenido).hexdigest()
    if ruta.exists():
        viejo = d2m.sha256_archivo(ruta)
        if viejo == nuevo:
            print(f"  [ya existe, identico] {ruta.name}")
            return nuevo
        raise FileExistsError(f"{ruta.name} ya existe con OTRO contenido "
                              f"(disco {viejo[:12]}..., nuevo {nuevo[:12]}...). "
                              "No se reescribe.")
    ruta.write_bytes(contenido)
    print(f"  [escrito] {ruta.name}  sha {nuevo[:12]}...")
    return nuevo

# --- 0. Rama ----------------------------------------------------------------
RAMA_NOMBRE = K.RAMA_OFICIAL_MED
RAMA = K.RAMAS_PENDIENTE_MED[RAMA_NOMBRE]
assert np.isclose(RAMA["sigma_z_m"], K.SIGMA_Z_ADOPTADO_MED_M), \
    "La rama oficial de Medellin y SIGMA_Z_ADOPTADO_MED_M no coinciden."
if RAMA["archivo"] is None:
    raise RuntimeError("Falta el nombre del CSV de pendientes en "
                       f"constantes_d1.RAMAS_PENDIENTE_MED['{RAMA_NOMBRE}']['archivo'].")
RUTA_PENDIENTE = K.ruta_cruda() / "recibidos" / RAMA["archivo"]
if not RUTA_PENDIENTE.exists():
    raise FileNotFoundError(f"Todavia no esta el archivo de pendiente:\n  {RUTA_PENDIENTE}")

SHA_PENDIENTE = d2m.sha256_archivo(RUTA_PENDIENTE)
if RAMA["sha256"] is None:
    print("REGISTRAR en constantes_d1.RAMAS_PENDIENTE_MED"
          f"['{RAMA_NOMBRE}']['sha256'] (reemplazar el None):")
    print(f'    "sha256": "{SHA_PENDIENTE}",')
    raise RuntimeError("Copie la linea de arriba en constantes_d1.py, reinicie el "
                       "kernel y vuelva a correr. (Se detiene a proposito.)")
assert SHA_PENDIENTE == RAMA["sha256"], \
    (f"El CSV de pendiente NO es el registrado.\n  registrado: {RAMA['sha256']}\n"
     f"  en disco  : {SHA_PENDIENTE}\nNo se sobrescriben entregas: guardela con otro nombre.")

CARPETA_SALIDA = f"D3_medellin/rama_{RAMA_NOMBRE}"
print(f"RAMA     : {RAMA_NOMBRE}  ({RAMA['descripcion']})")
print(f"sigma_z  : {RAMA['sigma_z_m']} m   ancho: {RAMA['ancho_correlacion_m']} m")
print(f"MDT      : {K.MDT_MED_PRODUCTO}  ({K.MDT_MED_CRS})")

# --- 1. El grafo es exactamente el que D2M registro ---------------------------
DIR_D2M = K.RUTA_ENTREGABLES / "D2_medellin"
RUTA_GPKG_D2M = DIR_D2M / "D2M_segmentos_para_pendiente.gpkg"
RUTA_HUE_D2M = DIR_D2M / "D2M_huellas_salidas.json"
RUTA_CONTRATO = DIR_D2M / "D2M_contrato_intercambio.json"
for _r in (RUTA_GPKG_D2M, RUTA_HUE_D2M, RUTA_CONTRATO):
    assert _r.exists(), f"Falta insumo: {_r}"

hue_d2m = json.loads(RUTA_HUE_D2M.read_text(encoding="utf-8"))
contrato = json.loads(RUTA_CONTRATO.read_text(encoding="utf-8"))
seg = gpd.read_file(RUTA_GPKG_D2M, layer=contrato.get("capa", "segmentos"))
seg["segmento_id"] = seg["segmento_id"].astype("string")
HUELLA_D2M = d2m.huella_segmentos(seg)
assert HUELLA_D2M == hue_d2m["huella_segmentos_gpkg"], \
    "El GeoPackage de D2M no es el que D2M registro. Vuelva a correr D2M."
assert HUELLA_D2M == contrato["huella_segmentos"], \
    "El contrato no corresponde a este grafo."
print(f"Grafo D2M: {len(seg):,} segmentos, huella verificada ({HUELLA_D2M[:12]}...)")

# Sentido de recorrido: D2M lo llama sentido_arista (Bogota: sentido_recorrido)
COL_SENTIDO = next(c for c in ("sentido_recorrido", "sentido_arista") if c in seg.columns)
for _c in ("id_eje", "parte", "longitud_m", "u", "v"):
    assert _c in seg.columns, f"Falta {_c} en el GPKG de D2M. Columnas: {list(seg.columns)}"
seg["id_eje"] = seg["id_eje"].astype("string")

# --- 2. El contrato pide las columnas que este cuaderno espera ----------------
_cols_contrato = list(contrato["columnas_retorno"])
if _cols_contrato != list(K.COLUMNAS_RETORNO_PENDIENTE_MED):
    print("AVISO: el contrato D2M lista otras columnas que constantes_d1:")
    print("  solo en contrato :", sorted(set(_cols_contrato) - set(K.COLUMNAS_RETORNO_PENDIENTE_MED)))
    print("  solo en constantes:", sorted(set(K.COLUMNAS_RETORNO_PENDIENTE_MED) - set(_cols_contrato)))
    print("  Se exige lo de constantes_d1 (incluye es_tunel). Corrija la celda 11 de D2M.")

pen = pd.read_csv(RUTA_PENDIENTE, dtype={"segmento_id": "string"},
                  encoding="utf-8", low_memory=False)
print(f"Pendiente: {len(pen):,} filas  sha {SHA_PENDIENTE[:12]}...")

BITACORA = {"cuaderno": "D3 Medellin — union grafo + pendiente",
            "sello_ejecucion": SELLO,
            "huella_segmentos_d2m": HUELLA_D2M,
            "archivo_pendiente": RAMA["archivo"], "sha256_pendiente": SHA_PENDIENTE,
            "rama": RAMA_NOMBRE, "rama_descripcion": RAMA["descripcion"],
            "sigma_z_rama": RAMA["sigma_z_m"],
            "ancho_correlacion_rama": RAMA["ancho_correlacion_m"],
            "mdt": K.MDT_MED_PRODUCTO, "mdt_crs": K.MDT_MED_CRS,
            "crs_red": str(seg.crs)}


RAMA     : med_sigma054  (Modulo de terreno (2026-11-11): sigma_z 0,539 m; ancho 59,2 m; v3, misma pendiente que v2, solo cambian error, marca 2 sigma y ruido)
sigma_z  : 0.539 m   ancho: 59.2 m
MDT      : DTM-LiDAR Medellín 2021, 1 m (Subsecretaría de Catastro)  (EPSG:6257)
Grafo D2M: 39,160 segmentos, huella verificada (5baf0bc9b8d2...)
Pendiente: 39,160 filas  sha 2aaba7230df0...


# Las pruebas del contrato

In [2]:
# =============================================================================
# D3M — CELDA 2: PRUEBAS DEL CONTRATO. Las 11 de Bogota, cada una recalculada
# por un camino independiente con la geometria de D2M, mas 2 de tuneles.
# =============================================================================
pr = []

# 1. Estan todas las columnas del contrato
_faltan = [c for c in K.COLUMNAS_RETORNO_PENDIENTE_MED if c not in pen.columns]
pr.append({"prueba": "columnas del contrato presentes (incluye es_tunel)",
           "faltan": _faltan, "pasa": not _faltan})
assert not _faltan, f"Faltan columnas del contrato: {_faltan}"

# 2. Los identificadores cruzan al 100 % en ambas direcciones, sin repetidos
_a, _b = set(seg["segmento_id"]), set(pen["segmento_id"])
pr.append({"prueba": "segmento_id: 100 % en ambas direcciones",
           "solo_en_grafo": len(_a - _b), "solo_en_pendiente": len(_b - _a),
           "repetidos_en_pendiente": int(pen["segmento_id"].duplicated().sum()),
           "pasa": _a == _b and pen["segmento_id"].is_unique})
assert _a == _b, "Los IDs no cruzan: el archivo no se hizo sobre esta exportacion de D2M."

m = seg[["segmento_id", "id_eje", "parte", COL_SENTIDO, "u", "v",
         "longitud_m"]].merge(pen, on="segmento_id", validate="one_to_one")
val = m["pendiente_neta_pct"].notna()

# 3. Pendientes coherentes con la LONGITUD DE D2M y en PORCENTAJE
_neta = 100 * (m["z_fin_m"] - m["z_inicio_m"]) / m["longitud_m"]
_acum = 100 * (m["ganancia_elevacion_m"] + m["perdida_elevacion_m"]) / m["longitud_m"]
_mal_neta = int((~np.isclose(m.loc[val, "pendiente_neta_pct"], _neta[val],
                             rtol=1e-6, atol=1e-6)).sum())
_mal_acum = int((~np.isclose(m["pendiente_acumulada_pct"], _acum,
                             rtol=1e-6, atol=1e-6, equal_nan=True)).sum())
pr.append({"prueba": "neta y acumulada = z y ganancia/perdida sobre longitud D2M",
           "n_neta_distinta": _mal_neta, "n_acumulada_distinta": _mal_acum,
           "pasa": _mal_neta == 0 and _mal_acum == 0})

# 4. FT y TF del mismo eje: signo opuesto, ganancia de uno = perdida del otro
_ft = m[m[COL_SENTIDO] == "FT"].set_index(["id_eje", "parte"])
_tf = m[m[COL_SENTIDO] == "TF"].set_index(["id_eje", "parte"])
_p = _ft.join(_tf, lsuffix="_ft", rsuffix="_tf", how="inner")
_ambos = _p["pendiente_neta_pct_ft"].notna() & _p["pendiente_neta_pct_tf"].notna()
_mal_signo = int((~np.isclose(_p.loc[_ambos, "pendiente_neta_pct_ft"],
                              -_p.loc[_ambos, "pendiente_neta_pct_tf"], atol=1e-6)).sum())
_mal_gan = int((~np.isclose(_p.loc[_ambos, "ganancia_elevacion_m_ft"],
                            _p.loc[_ambos, "perdida_elevacion_m_tf"], atol=1e-3)).sum())
_nan_asim = int((_p["pendiente_neta_pct_ft"].isna()
                 != _p["pendiente_neta_pct_tf"].isna()).sum())
pr.append({"prueba": "FT y TF del mismo eje: signo y subida/bajada simetricos",
           "n_pares": int(len(_p)), "n_signo_mal": _mal_signo,
           "n_ganancia_perdida_mal": _mal_gan, "n_vacio_asimetrico": _nan_asim,
           "pasa": _mal_signo == 0 and _mal_gan == 0 and _nan_asim == 0})

# 5. Sin pendiente => sin clase, sin error, sin marca 2 sigma, con bandera
_sin = ~val
_fuera = _a_bool(m["fuera_cobertura_mdt"])
_dc = _a_bool(m["distinta_de_cero_2sigma"])
_incoh = int((_sin & (m["clasificacion_invias"].notna()
                      | m["error_pendiente_pct"].notna() | _dc.notna())).sum())
_sin_band = int((_sin & (_fuera != True)).sum())
_band_dato = int((val & (_fuera == True)).sum())
pr.append({"prueba": "segmentos sin pendiente: coherentes y marcados",
           "n_sin_pendiente": int(_sin.sum()), "n_con_clase_o_error": _incoh,
           "n_sin_bandera": _sin_band, "n_bandera_con_dato": _band_dato,
           "pasa": _incoh == 0 and _sin_band == 0 and _band_dato == 0})

# 6. Clasificacion INVIAS recalculada desde la neta
_abs = m.loc[val, "pendiente_neta_pct"].abs()
_cls = np.select([_abs < 3, _abs < 6, _abs <= 8],
                 ["Plano", "Ondulado", "Montañoso"], "Escarpado")
_mal_cls = int((_cls != m.loc[val, "clasificacion_invias"].to_numpy()).sum())
pr.append({"prueba": "INVIAS = rangos oficiales sobre |neta|",
           "n_distinta": _mal_cls, "pasa": _mal_cls == 0})

# 7. sigma_z y ancho: un solo valor e iguales a los de la rama
_sz = m["sigma_z_m"].dropna().unique()
_an = m["ancho_correlacion_m"].dropna().unique()
pr.append({"prueba": f"sigma_z y ancho unicos e iguales a los de la rama {RAMA_NOMBRE}",
           "sigma_z_en_archivo": [float(x) for x in _sz],
           "ancho_en_archivo": [float(x) for x in _an],
           "sigma_z_rama": RAMA["sigma_z_m"], "ancho_rama": RAMA["ancho_correlacion_m"],
           "pasa": (len(_sz) == 1 and len(_an) == 1
                    and bool(np.isclose(_sz[0], RAMA["sigma_z_m"]))
                    and bool(np.isclose(_an[0], RAMA["ancho_correlacion_m"])))})

# 8. rho y error recalculados con la formula del contrato
if len(_an) == 1 and len(_sz) == 1:
    _rho = np.exp(-m["longitud_m"] / _an[0])
    _err = 100 * np.sqrt(2 * _sz[0] ** 2 * (1 - _rho)) / m["longitud_m"]
    _mal_rho = int((~np.isclose(m.loc[val, "rho_correlacion"], _rho[val], rtol=1e-6)).sum())
    _mal_err = int((~np.isclose(m.loc[val, "error_pendiente_pct"], _err[val], rtol=1e-6)).sum())
else:
    _mal_rho = _mal_err = -1
pr.append({"prueba": "rho y error = formula del contrato",
           "n_rho_distinto": _mal_rho, "n_error_distinto": _mal_err,
           "pasa": _mal_rho == 0 and _mal_err == 0})

# 9. La marca 2 sigma se recalcula
_dc_calc = m.loc[val, "pendiente_neta_pct"].abs() > 2 * m.loc[val, "error_pendiente_pct"]
_mal_dc = int((_dc_calc != _dc[val].astype(bool)).sum())
pr.append({"prueba": "distinta_de_cero_2sigma = |neta| > 2 * error",
           "n_distinta": _mal_dc, "pasa": _mal_dc == 0})

# 10. Piso de ruido de la acumulada, con la longitud de D2M
if len(_an) == 1 and len(_sz) == 1:
    _k = m["n_puntos_muestreados"] - 1
    _ok = _k.gt(0) & m["ruido_acumulada_pct"].notna()
    _p_real = m["longitud_m"] / _k
    _sd = _sz[0] * np.sqrt(2 * (1 - np.exp(-_p_real / _an[0])))
    _ruido = 100 * np.sqrt(2 / np.pi) * _sd / _p_real
    _mal_ruido = int((~np.isclose(m.loc[_ok, "ruido_acumulada_pct"], _ruido[_ok],
                                  rtol=1e-6)).sum())
else:
    _mal_ruido = -1
pr.append({"prueba": "ruido_acumulada_pct = formula del contrato",
           "n_distinto": _mal_ruido, "pasa": _mal_ruido == 0})

# 11. pendiente_extrema recalculada con la cota fisica
_mx = np.fmax(m["pendiente_neta_pct"].abs(), m["pendiente_acumulada_pct"].abs())
_ext_calc = _mx > K.UMBRAL_PENDIENTE_EXTREMA_PCT
_ext = _a_bool(m["pendiente_extrema"])
_hay = _mx.notna()
_mal_ext = int((_ext_calc[_hay] != _ext[_hay].astype(bool)).sum())
pr.append({"prueba": "pendiente_extrema = max(|neta|,|acumulada|) > cota",
           "n_extremos": int(_ext_calc[_hay].sum()), "n_distinto": _mal_ext,
           "pasa": _mal_ext == 0})

# 12. Tuneles: boca a boca (2 puntos) y acumulada = |neta|
m["es_tunel"] = _a_bool(m["es_tunel"])
_tun = m["es_tunel"] == True
_tun_val = _tun & val
_mal_np = int((m.loc[_tun_val, "n_puntos_muestreados"] != 2).sum())
_mal_tacum = int((~np.isclose(m.loc[_tun_val, "pendiente_acumulada_pct"],
                              m.loc[_tun_val, "pendiente_neta_pct"].abs(),
                              rtol=1e-6, atol=1e-6)).sum())
pr.append({"prueba": "tuneles: 2 puntos (bocas) y acumulada = |neta|",
           "n_tuneles": int(_tun.sum()), "n_sin_dato": int((_tun & ~val).sum()),
           "n_puntos_distinto_de_2": _mal_np, "n_acumulada_distinta": _mal_tacum,
           "pasa": m["es_tunel"].notna().all() and _mal_np == 0 and _mal_tacum == 0})

# 13. Tuneles: los dos sentidos de cada eje-parte marcados igual, y todo tramo
#     con "TUNEL" en el nombre de la Alcaldia esta marcado (minimo, no todos:
#     el de Oriente puede no llevarlo en el nombre)
_t = m.groupby(["id_eje", "parte"])["es_tunel"].nunique()
_mixto = int((_t > 1).sum())
_col_nom = next((c for c in ("nombre", "name") if c in seg.columns), None)
if _col_nom:
    _nom = seg.set_index("segmento_id")[_col_nom].reindex(m["segmento_id"]).to_numpy()
    _con_nombre = pd.Series(_nom).fillna("").map(d2m.normalizar).str.contains("TUNEL").to_numpy()
    _nombre_sin_marca = int((_con_nombre & ~_tun.to_numpy()).sum())
else:
    _nombre_sin_marca = -1
pr.append({"prueba": "tuneles: ambos sentidos iguales; 'TUNEL' en el nombre => marcado",
           "ejes_con_sentidos_distintos": _mixto,
           "con_nombre_tunel_sin_marca": _nombre_sin_marca,
           "pasa": _mixto == 0 and _nombre_sin_marca == 0})

RESUMEN = pd.DataFrame(pr)
RESUMEN["resultado"] = np.where(RESUMEN["pasa"], "PASA", "FALLA")
RESUMEN["detalle"] = [", ".join(f"{k}={v}" for k, v in p.items()
                                if k not in ("prueba", "pasa")) for p in pr]
print(RESUMEN[["prueba", "resultado", "detalle"]].to_string(index=False))
BITACORA["pruebas"] = pr
_fallan = RESUMEN.loc[~RESUMEN["pasa"], "prueba"].tolist()
assert not _fallan, f"NO se une. Fallan: {_fallan}"


                                                          prueba resultado                                                                                  detalle
              columnas del contrato presentes (incluye es_tunel)      PASA                                                                                faltan=[]
                         segmento_id: 100 % en ambas direcciones      PASA                           solo_en_grafo=0, solo_en_pendiente=0, repetidos_en_pendiente=0
      neta y acumulada = z y ganancia/perdida sobre longitud D2M      PASA                                                n_neta_distinta=0, n_acumulada_distinta=0
         FT y TF del mismo eje: signo y subida/bajada simetricos      PASA             n_pares=15755, n_signo_mal=0, n_ganancia_perdida_mal=0, n_vacio_asimetrico=0
                  segmentos sin pendiente: coherentes y marcados      PASA          n_sin_pendiente=2, n_con_clase_o_error=0, n_sin_bandera=0, n_bandera_con_dato=0
                

# La unión y las salidas

In [3]:
# =============================================================================
# D3M — CELDA 3: UNION Y SALIDAS (solo se llega aqui si las 13 pruebas pasan)
# Nada se reescribe: si un archivo existe con otro contenido, se detiene.
# =============================================================================
union = (seg.merge(pen, on="segmento_id", validate="one_to_one")
         .sort_values("segmento_id").reset_index(drop=True))
union["es_tunel"] = _a_bool(union["es_tunel"]).astype(bool)
union = gpd.GeoDataFrame(union, geometry="geometry", crs=seg.crs)

_n = f"_{RAMA_NOMBRE}"
sal_gpkg = K.ruta_entregable(CARPETA_SALIDA, f"D3M_grafo_con_pendiente{_n}.gpkg")
sal_csv  = K.ruta_entregable(CARPETA_SALIDA, f"D3M_grafo_con_pendiente{_n}.csv")
sal_hue  = K.ruta_entregable(CARPETA_SALIDA, f"D3M_huellas_salidas{_n}.json")
sal_bit  = K.ruta_entregable(CARPETA_SALIDA, f"bitacora_ejecucion_D3M{_n}_{SELLO}.json")

HUELLA_UNION = d2m.huella_segmentos(union)
assert HUELLA_UNION == HUELLA_D2M, "La geometria del grafo unido no es la de D2M"

# GPKG: el archivo cambia de bytes en cada escritura; se protege por huella
if sal_gpkg.exists():
    _prev = gpd.read_file(sal_gpkg, layer="grafo")
    _prev["segmento_id"] = _prev["segmento_id"].astype("string")
    assert d2m.huella_segmentos(_prev) == HUELLA_UNION and len(_prev) == len(union), \
        f"{sal_gpkg.name} ya existe con OTRO contenido. No se reescribe."
    print(f"  [ya existe, misma huella] {sal_gpkg.name}")
else:
    union.to_file(sal_gpkg, layer="grafo", driver="GPKG")
    _rel = gpd.read_file(sal_gpkg, layer="grafo")
    _rel["segmento_id"] = _rel["segmento_id"].astype("string")
    assert d2m.huella_segmentos(_rel) == HUELLA_D2M and len(_rel) == len(seg)
    print(f"  [escrito] {sal_gpkg.name}")

SHA_CSV = guardar_sin_sobrescribir(
    union.drop(columns="geometry").to_csv(index=False, lineterminator="\n").encode("utf-8"),
    sal_csv)

guardar_sin_sobrescribir(json.dumps({
    "archivos": {sal_csv.name: SHA_CSV},
    "contenido": {sal_gpkg.name: HUELLA_UNION},
    "definicion_huella": "sha256 sobre (segmento_id + WKB) ordenado por segmento_id",
    "rama": RAMA_NOMBRE,
    "sigma_z_m": RAMA["sigma_z_m"],
    "ancho_correlacion_m": RAMA["ancho_correlacion_m"],
    "mdt": K.MDT_MED_PRODUCTO, "mdt_crs": K.MDT_MED_CRS,
    "insumos": {"huella_segmentos_d2m": HUELLA_D2M,
                "archivo_pendiente": RAMA["archivo"],
                "sha256_pendiente": SHA_PENDIENTE},
}, indent=2, ensure_ascii=False).encode("utf-8"), sal_hue)

BITACORA["ejecucion_terminada"] = datetime.now(ZoneInfo("America/Bogota")).isoformat(timespec="seconds")
sal_bit.write_text(json.dumps(BITACORA, indent=2, ensure_ascii=False, default=str),
                   encoding="utf-8")

print(f"\nGrafo con pendiente: {len(union):,} segmentos")
print(f"  sin pendiente (fuera de cobertura): {int(union['pendiente_neta_pct'].isna().sum()):,}")
print(f"  tuneles: {int(union['es_tunel'].sum()):,} aristas")
for p in (sal_gpkg, sal_csv, sal_hue, sal_bit):
    print(f"  {p.name:<48} {p.stat().st_size/1e6:>8.2f} MB")


  [escrito] D3M_grafo_con_pendiente_med_sigma054.gpkg
  [escrito] D3M_grafo_con_pendiente_med_sigma054.csv  sha 845158ec00ad...
  [escrito] D3M_huellas_salidas_med_sigma054.json  sha aff932b7147e...

Grafo con pendiente: 39,160 segmentos
  sin pendiente (fuera de cobertura): 2
  tuneles: 30 aristas
  D3M_grafo_con_pendiente_med_sigma054.gpkg           17.33 MB
  D3M_grafo_con_pendiente_med_sigma054.csv            12.94 MB
  D3M_huellas_salidas_med_sigma054.json                0.00 MB
  bitacora_ejecucion_D3M_med_sigma054_20261112_095641.json     0.00 MB


# Diagnóstico para la revisión (no decide nada)

Responde las preguntas del módulo de terreno con los datos ya unidos.
Escribe una lista para revisar en Street View; no cambia el grafo ni la pendiente.

In [4]:
# =============================================================================
# D3M — CELDA 4: DIAGNOSTICO (solo lee lo ya unido; escribe una lista de revision)
# =============================================================================
U = union.copy()
U["pendiente_extrema"] = _a_bool(U["pendiente_extrema"]).fillna(False).astype(bool)
_col_jer = next((c for c in ("jerarquia", "jerarquia_op") if c in U.columns), None)
U["paso_nivel_num"] = pd.to_numeric(U.get("paso_nivel"), errors="coerce")
U["es_elevado"] = _a_bool(U["es_elevado"]).fillna(False).astype(bool) \
    if "es_elevado" in U.columns else False

def _calles(mascara):
    """Cuenta calles fisicas (id_eje, parte), no sentidos."""
    return U.loc[mascara, ["id_eje", "parte"]].drop_duplicates().shape[0]

ext = U["pendiente_extrema"]
sin_jer = U[_col_jer].isna() if _col_jer else pd.Series(False, index=U.index)
print(f"Aristas con pendiente > {K.UMBRAL_PENDIENTE_EXTREMA_PCT:g} %: {int(ext.sum()):,} "
      f"({_calles(ext):,} calles)")
print(f"  de ellas sin jerarquia : {int((ext & sin_jer).sum()):,} ({_calles(ext & sin_jer):,} calles)")
print(f"  de ellas tuneles       : {int((ext & U['es_tunel']).sum()):,} (deberia ser 0)")
print(f"  de ellas elevadas      : {int((ext & U['es_elevado']).sum()):,}")
if _col_jer:
    print("\nExtremas por jerarquia (aristas):")
    print(U.loc[ext, _col_jer].fillna("(vacia)").value_counts().to_string())

print(f"\npaso_nivel = 2: {int((U['paso_nivel_num'] == 2).sum())} aristas "
      f"(es_elevado=True en {int((U['es_elevado'] & (U['paso_nivel_num'] == 2)).sum())})")
print(f"es_elevado    : {int(U['es_elevado'].sum()):,} aristas")
print(f"Tuneles       : {int(U['es_tunel'].sum()):,} aristas, "
      f"{U.loc[U['es_tunel'], 'longitud_m'].sum()/1000:.2f} km (sumando ambos sentidos)")

# Lista para Street View: una fila por calle fisica, el sentido que sube.
# Extremas + paso_nivel 2. Coordenadas del punto medio en WGS84.
rev = U[ext | (U["paso_nivel_num"] == 2)].copy()
rev["motivo"] = np.where(rev["pendiente_extrema"], "extrema", "paso_nivel_2")
rev["sin_jerarquia"] = rev[_col_jer].isna() if _col_jer else np.nan
rev = (rev.sort_values("pendiente_neta_pct", ascending=False)
          .drop_duplicates(["id_eje", "parte"]))
_mid = rev.geometry.interpolate(0.5, normalized=True).to_crs("EPSG:4326")
rev["lat"], rev["lon"] = _mid.y.round(6), _mid.x.round(6)
rev["street_view"] = ("https://www.google.com/maps/@?api=1&map_action=pano&viewpoint="
                      + rev["lat"].astype(str) + "," + rev["lon"].astype(str))
_cols = ["segmento_id", "id_eje", "parte", "motivo", "sin_jerarquia",
         _col_nom, _col_jer, "paso_nivel", "es_elevado", "longitud_m",
         "pendiente_neta_pct", "pendiente_acumulada_pct", "lat", "lon", "street_view"]
rev = rev[[c for c in _cols if c and c in rev.columns]]
rev["se_ve_calle_para_carros"] = ""      # lo llena quien revisa: si / no / no_se
rev["observacion"] = ""
_ruta_rev = K.ruta_entregable(CARPETA_SALIDA, f"D3M_revision_street_view{_n}.csv")
guardar_sin_sobrescribir(rev.to_csv(index=False, lineterminator="\n").encode("utf-8"),
                         _ruta_rev)
print(f"\nLista para Street View: {len(rev):,} calles -> {_ruta_rev.name}")


Aristas con pendiente > 35 %: 189 (95 calles)
  de ellas sin jerarquia : 103 (52 calles)
  de ellas tuneles       : 1 (deberia ser 0)
  de ellas elevadas      : 3

Extremas por jerarquia (aristas):
jerarquia
(vacia)                               103
5-Servicio                             78
7-Vía Rural (Vías inter-veredales)      6
4-Colectora                             2

paso_nivel = 2: 11 aristas (es_elevado=True en 1)
es_elevado    : 493 aristas
Tuneles       : 30 aristas, 20.09 km (sumando ambos sentidos)
  [escrito] D3M_revision_street_view_med_sigma054.csv  sha 60d106cbcfaa...

Lista para Street View: 104 calles -> D3M_revision_street_view_med_sigma054.csv
